# Stage 4 · Heterogeneity, Alternative Outcomes, and Falsification

Estimates subgroup effects with per-subgroup power, alternative outcome
margins (intensity, delay, abandonment, sustained activity), the
registry-removal analysis, and the placebo-timing falsification
(paper Secs. 4.3-4.5). Runs entirely from cached state; no new collection.

---
## Cell 0 — Crash-safe infrastructure (run first, every session)

In [ ]:
# ============ CELL 0 — CRASH-SAFE INFRASTRUCTURE ============
"""
Crash-safe collection primitives for the MSR 2027 pipeline.

Design rules, each learned from a way Colab actually fails:
  1. APPEND-ONLY JSONL, never rewrite-in-place. A crash mid-write loses at
     most the last partial line, not the whole file.
  2. ATOMIC writes for any file that must be rewritten (tmp + os.replace).
  3. FAILURES ARE RECORDED, so permanently-dead items are not retried forever.
  4. MIRRORED to Drive but WRITTEN to local disk, so a dropped Drive mount
     mid-run cannot corrupt the checkpoint.
  5. RESUME IS O(n), not O(n^2).
"""
from __future__ import annotations
import json, os, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor


def atomic_write_bytes(path: Path, data: bytes) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


class ResumableCollector:
    """Append-only, crash-safe, resumable map over a list of string keys."""

    def __init__(self, name, local_dir='/content/msr_state', mirror_dir=None,
                 workers=12, flush_every=25, mirror_every=100, mirror_seconds=60, max_attempts=2):
        self.name = name
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.mirror = Path(mirror_dir) if mirror_dir else None
        if self.mirror: self.mirror.mkdir(parents=True, exist_ok=True)
        self.workers, self.flush_every = workers, flush_every
        self.mirror_every, self.max_attempts = mirror_every, max_attempts
        self.mirror_seconds = mirror_seconds
        self._last_mirror = time.time()
        self.data_path = self.local/f'{name}.jsonl'
        self.fail_path = self.local/f'{name}.failed.jsonl'
        self._restore_from_mirror()
        self.done, self.failed = self._load()

    # ---- mirror handling -------------------------------------------------
    def _restore_from_mirror(self):
        """If local disk was wiped (new runtime) but Drive has state, pull it back."""
        if not self.mirror: return
        for p in (self.data_path, self.fail_path):
            m = self.mirror/p.name
            if m.exists() and (not p.exists() or m.stat().st_size > p.stat().st_size):
                shutil.copy2(m, p)

    def _push_mirror(self):
        if not self.mirror: return
        try:
            for p in (self.data_path, self.fail_path):
                if p.exists(): shutil.copy2(p, self.mirror/p.name)
        except Exception as e:
            print(f'  [mirror warning: {e}]')   # never fatal

    # ---- state -----------------------------------------------------------
    def _load(self):
        done, failed = {}, {}
        if self.data_path.exists():
            with open(self.data_path) as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        done[r['_key']] = r
                    except Exception:
                        continue          # truncated final line after a crash
        if self.fail_path.exists():
            with open(self.fail_path) as f:
                for line in f:
                    try:
                        r = json.loads(line); failed[r['_key']] = r.get('n', 1)
                    except Exception:
                        continue
        return done, failed

    def pending(self, keys):
        """O(n) resume: skip completed and permanently-failed keys."""
        d, fl, m = self.done, self.failed, self.max_attempts
        return [k for k in keys if k not in d and fl.get(k, 0) < m]

    # ---- the run loop ----------------------------------------------------
    def run(self, keys, fn, progress=True):
        """Workers write their OWN results the moment they complete.

        The earlier version wrote from the consumer side of ThreadPoolExecutor.map().
        That is wrong: map() submits everything eagerly, so workers can finish
        thousands of items while the consumer lags behind, and every one of those
        is lost on a hard kill. Writing inside the worker, under a lock, means a
        completed item is durable the moment it exists.
        """
        import threading
        todo = self.pending(keys)
        print(f'[{self.name}] total={len(keys):,}  done={len(self.done):,}  '
              f'failed={len(self.failed):,}  todo={len(todo):,}')
        if not todo:
            print(f'[{self.name}] nothing to do — already complete.')
            return self.frame()

        lock = threading.Lock()
        fd = open(self.data_path, 'a'); ff = open(self.fail_path, 'a')
        counters = {'ok': 0, 'bad': 0, 'since_flush': 0, 'since_mirror': 0}
        t0 = time.time()

        def worker(key):
            try:
                rec = fn(key)
            except Exception:
                rec = None
            with lock:
                if rec is None:
                    counters['bad'] += 1
                    n = self.failed.get(key, 0) + 1
                    ff.write(json.dumps({'_key': key, 'n': n}) + '\n')
                    self.failed[key] = n
                else:
                    counters['ok'] += 1
                    r = dict(rec); r['_key'] = key
                    fd.write(json.dumps(r, default=str) + '\n')
                    self.done[key] = r
                counters['since_flush'] += 1
                counters['since_mirror'] += 1
                if counters['since_flush'] >= self.flush_every:
                    fd.flush(); os.fsync(fd.fileno())
                    ff.flush(); os.fsync(ff.fileno())
                    counters['since_flush'] = 0
                due = (counters['since_mirror'] >= self.mirror_every or
                       time.time() - self._last_mirror >= self.mirror_seconds)
                if due:
                    fd.flush(); ff.flush(); self._push_mirror()
                    counters['since_mirror'] = 0; self._last_mirror = time.time()
            return key

        try:
            from tqdm.auto import tqdm
            wrap = (lambda it: tqdm(it, total=len(todo))) if progress else (lambda it: it)
        except Exception:
            wrap = lambda it: it

        try:
            with ThreadPoolExecutor(max_workers=self.workers) as ex:
                for _ in wrap(ex.map(worker, todo)):
                    pass
        except KeyboardInterrupt:
            print(f'\n[{self.name}] INTERRUPTED — progress is safe. Re-run to resume.')
        finally:
            try:
                fd.flush(); os.fsync(fd.fileno()); fd.close()
                ff.flush(); os.fsync(ff.fileno()); ff.close()
            except Exception: pass
            self._push_mirror()

        print(f'[{self.name}] +{counters["ok"]:,} ok, +{counters["bad"]:,} failed '
              f'in {time.time()-t0:.0f}s -> {len(self.done):,} total')
        return self.frame()

    def frame(self):
        import pandas as pd
        if not self.done: return pd.DataFrame()
        df = pd.DataFrame(list(self.done.values()))
        return df.rename(columns={'_key': 'project'})


class CellState:
    """Records which cells completed, so a fresh runtime tells you where you were."""
    def __init__(self, path):
        self.path = Path(path); self.state = {}
        if self.path.exists():
            try: self.state = json.loads(self.path.read_text())
            except Exception: self.state = {}

    def mark(self, cell):
        self.state[str(cell)] = time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime())
        atomic_write_bytes(self.path, json.dumps(self.state, indent=2).encode())

    def report(self):
        if not self.state:
            print('No previous run recorded — start at cell 1.'); return
        print('Previously completed cells:')
        for k in sorted(self.state, key=lambda x: int(x)):
            print(f'   cell {k}: {self.state[k]}')
        nxt = max(int(k) for k in self.state) + 1
        print(f'-> After a disconnect, re-run cells 1..{nxt-1} (they are cached '
              f'and fast), then continue at cell {nxt}.')

---
## Cell 1 — Setup

Rebuilds the analysis frame and panel entirely from Drive caches. The collector recognises all keys as done, so this touches no network.

In [ ]:
# ============================ CELL 1 — SETUP ============================
if 'ResumableCollector' not in globals(): raise SystemExit('>>> Run CELL 0 first.')
!pip -q install linearmodels rdrobust pandas numpy scipy matplotlib pyarrow tqdm 2>/dev/null

import os, json, time, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
from rdrobust import rdrobust
from scipy import stats

try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/msr2027')
except Exception:
    WORK = Path('./msr2027')
CACHE = WORK/'cache'
STATE = CellState(CACHE/'checkpoint_stage4.json'); STATE.report()

T0 = pd.Timestamp(2022,7,8)
PANEL_START, PANEL_END = '2021-01', '2023-12'

full = pd.read_parquet(CACHE/'analysis_frame_rdd.parquet')
late = set(pd.read_parquet(CACHE/'late_entrants.parquet')['project'])
frame = full[~full['project'].isin(late)].copy()
d3 = json.loads((CACHE/'results_panel.json').read_text())

COL = ResumableCollector('pypi_month_panel', local_dir='/content/msr_state',
                         mirror_dir=str(CACHE/'state'), workers=12)
gotm = COL.frame()
if len(gotm)==0: raise SystemExit('>>> Panel collector state not found in Drive cache (run notebook 03 first).')

months = pd.period_range(PANEL_START, PANEL_END, freq='M').astype(str).tolist()
t0i = months.index('2022-07')
mset = {p:set(m) for p,m in zip(gotm['project'], gotm['months'])}
tr   = dict(zip(frame['project'], frame['treated']))

rows=[]
for p in frame['project']:
    if p not in mset: continue
    s=mset[p]; t=tr[p]
    for i,mm in enumerate(months):
        rows.append((p, i, mm, int(mm in s), int(t)))
panel = pd.DataFrame(rows, columns=['project','mnum','month','released','treated'])
panel['post'] = (panel['mnum']>=t0i).astype(int)
panel['D'] = (panel['treated']*panel['post']).astype(float)

def did(sub):
    """TWFE DiD on a project subset -> (att, se, p, mde, n_proj)."""
    pp = panel[panel['project'].isin(sub)].set_index(['project','mnum'])
    if pp.reset_index()['project'].nunique() < 80: return None
    m = PanelOLS(pp['released'], pp[['D']], entity_effects=True, time_effects=True
        ).fit(cov_type='clustered', cluster_entity=True)
    att, se = float(m.params['D']), float(m.std_errors['D'])
    mde = (stats.norm.ppf(0.975)+stats.norm.ppf(0.80))*se
    return att, se, float(m.pvalues['D']), mde, pp.reset_index()['project'].nunique()

BASE = panel.loc[(panel.treated==0)&(panel.post==0),'released'].mean()
print(f'Panel rebuilt: {len(panel):,} rows, {panel.project.nunique():,} projects; base rate {BASE:.4f}')
STATE.mark(1); SETUP_DONE=True; print('\nCELL 1 OK')

---
## Cell 2 — Heterogeneity: the projects the burden argument is about

Splits defined on **pre-treatment** characteristics only (anything measured post-treatment is itself an outcome). For each subgroup: ATT, p, and — just as important — the subgroup MDE, because "null in small projects" only counts if we had the power to see an effect there.

In [ ]:
# ==================== CELL 2 (FIXED) — HETEROGENEITY ====================
# Fix 1: guard did() against subgroups with no treated/control contrast.
# Fix 2: drop the download-volume split — within the band, downloads ARE the
#         treatment assignment (the cutoff is a download rank), so splitting
#         on them just re-splits treatment: no valid comparison exists.
if not globals().get('SETUP_DONE'): raise SystemExit('>>> Run CELL 1 first.')

def did(sub):
    pp = panel[panel['project'].isin(sub)]
    n_t = pp.loc[pp.treated==1,'project'].nunique()
    n_c = pp.loc[pp.treated==0,'project'].nunique()
    if n_t < 40 or n_c < 40:
        return ('no-contrast', n_t, n_c)
    pp = pp.set_index(['project','mnum'])
    m = PanelOLS(pp['released'], pp[['D']], entity_effects=True, time_effects=True
        ).fit(cov_type='clustered', cluster_entity=True)
    att, se = float(m.params['D']), float(m.std_errors['D'])
    mde = (stats.norm.ppf(0.975)+stats.norm.ppf(0.80))*se
    return att, se, float(m.pvalues['D']), mde, n_t+n_c

fr = frame.copy()
fr['low_activity'] = (fr['n_rel_pre'] <= fr['n_rel_pre'].median()).astype(int)
fr['old']          = (fr['age_days']  >= fr['age_days'].median()).astype(int)
fr['dormant_pre']  = (fr['n_rel_pre'] == 0).astype(int)

splits = {
 'ALL (panel headline)'        : fr['project'],
 'low pre-activity (<=median)' : fr.loc[fr.low_activity==1,'project'],
 'high pre-activity'           : fr.loc[fr.low_activity==0,'project'],
 'dormant pre-period (0 rel.)' : fr.loc[fr.dormant_pre==1,'project'],
 'older projects (>=median)'   : fr.loc[fr.old==1,'project'],
 'younger projects'            : fr.loc[fr.old==0,'project'],
 'old AND low-activity'        : fr.loc[(fr.old==1)&(fr.low_activity==1),'project'],
 'young AND dormant'           : fr.loc[(fr.old==0)&(fr.dormant_pre==1),'project'],
}
print('='*84)
print(f'{"subgroup":30s} {"ATT":>9s} {"p":>7s} {"MDE":>8s} {"MDE/base":>9s} {"n_proj":>7s}')
print('='*84)
HET={}
for name, sub in splits.items():
    r = did(set(sub))
    if r[0]=='no-contrast':
        print(f'{name:30s}  skipped: insufficient contrast '
              f'({r[1]} treated / {r[2]} control)'); continue
    att,se,p,mde,n = r
    HET[name]=dict(att=att,se=se,p=p,mde=mde,n=n)
    print(f'{name:30s} {att:+9.4f} {p:7.3f} {mde:8.4f} {mde/BASE:8.1%} {n:7d}')
print('='*84)
sig=[k for k,v in HET.items() if v['p']<0.05 and k!='ALL (panel headline)']
print(f'\nSubgroups significant at 5%: {len(sig)}/{max(len(HET)-1,1)}'
      + (f'  -> {sig}  INVESTIGATE before writing' if sig else '  -> uniform null'))
STATE.mark(2); HET_DONE=True; print('\nCELL 2 OK')

---
## Cell 3 — Alternative outcomes

Closes the "your binary outcome hid the slowdown" objection. Three angles: **intensity** (monthly release counts, winsorised — did projects release *less often* even if not never), **delay** (RDD on days-to-next-release, from stage 2), and **abandonment** (RDD on the 2-year abandonment flag).

In [ ]:
# ================= CELL 3 — ALTERNATIVE OUTCOMES ========================
if not globals().get('HET_DONE'): raise SystemExit('>>> Run CELL 2 first.')

# (a) intensity and abandonment via RDD on the 12-month cross-section.
# (The stage-3 panel stores month PRESENCE, not counts, so monthly-count DiD would
#  need another collection pass; log1p(n_rel_post) via RDD covers intensity.)
ALT={}
d2f = full.copy()
def rdd(y, x):
    y=np.asarray(y,float); x=np.asarray(x,float)
    ok=np.isfinite(y)&np.isfinite(x)
    r=rdrobust(y=y[ok], x=x[ok], c=0)
    return float(r.coef.iloc[0,0]), float(r.pv.iloc[2,0]), int(np.sum(np.asarray(r.N_h)))

for nm, col in [('log1p releases (12m post)', np.log1p(d2f['n_rel_post'])),
                ('days to next release',       d2f['days_to_next']),
                ('abandoned within 2 years',   d2f['abandoned'])]:
    e,p,n = rdd(col, d2f['running'])
    ALT[nm]=dict(est=e,p=p,n=n)
    print(f'  RDD {nm:28s} est={e:+9.4f}  p={p:.3f}  n_eff={n}')

# (b) DiD on a stricter binary: released in >=2 of trailing 3 months ("sustained activity")
pan3 = panel.sort_values(['project','mnum']).copy()
pan3['roll3'] = (pan3.groupby('project')['released']
                     .transform(lambda s: s.rolling(3, min_periods=3).sum()))
pan3 = pan3.dropna(subset=['roll3'])
pan3['sustained'] = (pan3['roll3']>=2).astype(float)
pp = pan3.set_index(['project','mnum'])
m = PanelOLS(pp['sustained'], pp[['D']], entity_effects=True, time_effects=True
    ).fit(cov_type='clustered', cluster_entity=True)
ALT['sustained activity (DiD)'] = dict(est=float(m.params['D']),
                                       p=float(m.pvalues['D']),
                                       n=int(pp.reset_index()['project'].nunique()))
print(f'  DiD sustained activity           est={float(m.params["D"]):+9.4f}  '
      f'p={float(m.pvalues["D"]):.3f}')
STATE.mark(3); ALT_DONE=True; print('\nCELL 3 OK')

---
## Cell 3b — Protest deletions: did atomicwrites have silent company?

The most famous casualty of this policy was `atomicwrites`: its maintainer deleted it from PyPI in July 2022 in direct protest of the critical-project designation. The whole policy debate has run on that anecdote. This cell asks the population question: **were designated projects removed from PyPI at a higher rate than near-cutoff controls?**

The outcome is already in the RDD-stage collector state — projects that 404ed were recorded as `removed=1`, then dropped from the analysis frame (correctly, for release outcomes). Here they come back as the outcome itself. RDD on removal, plus a raw count on each side of the cutoff, since removals are rare events and the raw numbers may be more informative than the estimate.

In [ ]:
# ============ CELL 3b (FIXED) — PROTEST DELETIONS (REMOVAL OUTCOME) ============
# Fix 1: the stage-2 frame already has a 'removed' column (all zeros) -> merge
#        collision. Fix 2 (the real one): removed projects were FILTERED OUT of
#        that frame before it was saved, so they cannot be recovered from it.
#        Rebuild the sample from the ranking cache, where every project exists.
if not globals().get('ALT_DONE'): raise SystemExit('>>> Run CELL 3 first.')

COL2 = ResumableCollector('pypi_outcomes', local_dir='/content/msr_state',
                          mirror_dir=str(CACHE/'state'), workers=1)
raw2 = COL2.frame()
if 'removed' not in raw2.columns:
    print('No removal flags in stage-2 state — skip.'); REMOVAL={'available': False}
else:
    rankdf = pd.read_parquet(sorted(CACHE.glob('ranking_*.parquet'))[-1])
    CUT_DL = float(rankdf.loc[rankdf['rank']==3800, 'num_downloads'].iloc[0])

    rem = raw2[['project','removed']].copy()
    rem['removed'] = pd.to_numeric(rem['removed'], errors='coerce').fillna(0).astype(int)

    # Everything the stage-2 collector touched WAS in the clean band; rebuild
    # treated/running from the ranking, keep the late-entrant exclusion.
    band_r = rankdf.merge(rem, on='project', how='inner')
    band_r = band_r[~band_r['project'].isin(late)]
    band_r['treated'] = (band_r['rank'] <= 3800).astype(int)
    band_r['running'] = np.log(band_r['num_downloads']) - np.log(CUT_DL)

    n_t = int(band_r.loc[band_r.treated==1,'removed'].sum())
    N_t = int((band_r.treated==1).sum())
    n_c = int(band_r.loc[band_r.treated==0,'removed'].sum())
    N_c = int((band_r.treated==0).sum())
    print(f'Removed from PyPI (as of collection date):')
    print(f'  treated {n_t}/{N_t} ({n_t/max(N_t,1):.3%})   '
          f'control {n_c}/{N_c} ({n_c/max(N_c,1):.3%})')

    from scipy.stats import fisher_exact
    odds, p_f = fisher_exact([[n_t, N_t-n_t],[n_c, N_c-n_c]])
    print(f'  Fisher exact: odds ratio {odds:.2f}, p = {p_f:.4f}')
    REMOVAL = {'available': True, 'treated': [n_t,N_t], 'control': [n_c,N_c],
               'fisher_or': float(odds), 'fisher_p': float(p_f)}

    if (n_t + n_c) >= 20:
        e,pv,n = rdd(band_r['removed'], band_r['running'])
        print(f'  RDD on removal: est={e:+.4f}  p={pv:.3f}  n_eff={n}')
        REMOVAL.update(rdd_est=float(e), rdd_p=float(pv))
    else:
        print(f'  Only {n_t+n_c} removals in the whole band — too few for RDD;')
        print('  the Fisher counts ARE the result: near-cutoff removals were')
        print('  vanishingly rare. Report the raw counts.')

    # Caveats to carry into the paper, stated now so they are not forgotten:
    # (1) removal is measured as of the collection date (2026), not within a
    #     fixed post-window — it is cumulative, and removal DATES are unknown;
    # (2) a 404 on both APIs is the detection rule; PyPI rarely hard-deletes,
    #     and atomicwrites itself was later re-created, so this undercounts
    #     temporary protest deletions and catches only lasting ones.
STATE.mark(35); REMOVAL_DONE=True; print('\nCELL 3b OK')

---
## Cell 4 — Placebo timing test

Pretend designation happened in **July 2021**, using only genuinely pre-treatment data (through June 2022). Any machinery that produces an "effect" where no policy existed is broken machinery. This should be a flat zero.

In [ ]:
# ================== CELL 4 — PLACEBO TIMING =============================
if not globals().get('REMOVAL_DONE'): raise SystemExit('>>> Run CELL 3 first.')

fake_i = months.index('2021-07')
pre = panel[panel['mnum'] < t0i].copy()          # only pre-treatment months
pre['Dfake'] = (pre['treated']*(pre['mnum']>=fake_i)).astype(float)
pp = pre.set_index(['project','mnum'])
mF = PanelOLS(pp['released'], pp[['Dfake']], entity_effects=True, time_effects=True
     ).fit(cov_type='clustered', cluster_entity=True)
P_ATT, P_P = float(mF.params['Dfake']), float(mF.pvalues['Dfake'])
print(f'  Placebo (fake designation 2021-07, pre-data only):')
print(f'  "ATT" = {P_ATT:+.5f}   p = {P_P:.4f}')
print('  ' + ('PASS — machinery produces no effect where none exists.'
              if P_P>=0.05 else
              'FAIL — pre-period divergence; the DiD identifying assumption is suspect.'))
STATE.mark(4); PLACEBO_DONE=True; print('\nCELL 4 OK')

---
## Cell 5 — Consolidated export


In [ ]:
# ===================== CELL 5 — EXPORT ==================================
if not globals().get('PLACEBO_DONE'): raise SystemExit('>>> Run CELL 4 first.')
out = {
 'run_at': dt.datetime.now(dt.timezone.utc).isoformat(),
 'headline': d3,
 'base_monthly_rate': float(BASE),
 'heterogeneity': HET,
 'alternative_outcomes': ALT,
 'placebo_timing': {'att': P_ATT, 'p': P_P},
 'removal_outcome': REMOVAL,
}
(CACHE/'results_extended.json').write_text(json.dumps(out, indent=2, default=float))
print(json.dumps(out, indent=2, default=float)[:3000])
print('\nSaved results_extended.json.')